In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.visualisation import configure_mpl

configure_mpl(Path("../fonts/"))

RANDOM_SEED = 202607101941

DATA_PATH = Path("../reports/thesis/results/data/model/all_interventions/")
schema = schema.post_index()

rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
labels = np.asarray(schema.get_short_names("measurement"))

In [ ]:
null_measurements_asym = np.load(DATA_PATH / "ising_00.npz")["measurements"][
    ..., 5, :, 7
]
null_measurements_sym = np.load(DATA_PATH / "sym_ising_00.npz")["measurements"][
    ..., 5, :, 7
]

In [ ]:
collective_effect = []

In [ ]:
for delta in ("05", "10", "25"):
    measurements_asym = np.load(DATA_PATH / f"ising_{delta}.npz")["measurements"][
        ..., 5, :, 7
    ]
    measurements_sym = np.load(DATA_PATH / f"sym_ising_{delta}.npz")["measurements"][
        ..., 5, :, 7
    ]
    asym_effect = measurements_asym - null_measurements_asym
    sym_effect = measurements_sym - null_measurements_sym
    effect_of_asymmetry = asym_effect - sym_effect
    collective_effect.append(effect_of_asymmetry.mean(axis=1))

In [ ]:
collective_effect = np.asarray(collective_effect)

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 2.75), constrained_layout=True)

mean_collective_effect = collective_effect.mean(axis=1)
sort_idx = np.argsort(mean_collective_effect[0])[::-1]

means = mean_collective_effect[:, sort_idx]
plot_labels = labels[sort_idx]

plot_df = (
    pl.DataFrame(means.T, schema=["Weak", "Medium", "Strong"])
    .with_columns(Spin=plot_labels)
    .unpivot(index="Spin", variable_name="Scenario", value_name="Effect of Asymmetry")
)

sns.barplot(plot_df, x="Spin", y="Effect of Asymmetry", hue="Scenario", ax=ax)
ax.set_xticks(
    np.arange(len(plot_labels)), plot_labels, rotation=45, horizontalalignment="right"
)

ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)
ax.set_ylabel("Effect of asymmetry");